In [ ]:
#| default_exp memory

In [ ]:
#| include: false
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import warnings
from dataclasses import dataclass, asdict
from typing import Sequence
from fasterbench.core import _bytes_to_mib, _run_on_devices, _ensure_device_supported

import numpy as np
import torch

try:
    import psutil
except ImportError:
    psutil = None

In [ ]:
#| export
@dataclass(slots=True)
class MemoryMetrics:
    """Memory usage metrics for a single device."""
    avg_mib: float
    peak_mib: float
    reserved_mib: float  # GPU-only (NaN for CPU)
    device: str

    def as_dict(self) -> dict[str, float | str]:
        return asdict(self)


#| export
def _nan_memory_metrics(device: str) -> MemoryMetrics:  # device string
    """Create MemoryMetrics with NaN values for failed benchmarks."""
    nan = float("nan")
    return MemoryMetrics(nan, nan, nan, device)


#| export
def _gpu_metrics(
    model: torch.nn.Module,    # model to benchmark
    sample: torch.Tensor,      # input tensor (with batch dimension)
    *,
    warmup: int,               # warmup iterations
    steps: int,                # measurement iterations
) -> MemoryMetrics:
    """Measure GPU memory usage."""
    dev = torch.device("cuda")
    _ensure_device_supported(model, dev)  # quantized ops are CPU-only; avoid SIGSEGV
    model = model.eval().to(dev)
    sample = sample.to(dev)

    for _ in range(warmup):
        model(sample)
    torch.cuda.synchronize(dev)

    alloc, alloc_peak, reserv_peak = [], [], []
    for _ in range(steps):
        torch.cuda.reset_peak_memory_stats(dev)
        model(sample)
        torch.cuda.synchronize(dev)
        alloc.append(torch.cuda.memory_allocated(dev))
        alloc_peak.append(torch.cuda.max_memory_allocated(dev))
        reserv_peak.append(torch.cuda.max_memory_reserved(dev))

    return MemoryMetrics(
        avg_mib=_bytes_to_mib(float(np.mean(alloc))),
        peak_mib=_bytes_to_mib(float(np.mean(alloc_peak))),
        reserved_mib=_bytes_to_mib(float(np.mean(reserv_peak))),
        device="cuda",
    )


#| export
def _cpu_metrics(
    model: torch.nn.Module,    # model to benchmark
    sample: torch.Tensor,      # input tensor (with batch dimension)
    *,
    warmup: int,               # warmup iterations
    steps: int,                # measurement iterations
) -> MemoryMetrics:
    """Measure CPU memory usage via psutil."""
    if psutil is None:
        warnings.warn("psutil not available – returning NaNs for CPU memory")
        return _nan_memory_metrics("cpu")

    proc = psutil.Process()
    model = model.eval().cpu()
    sample = sample.cpu()

    rss0 = proc.memory_info().rss

    for _ in range(warmup):
        model(sample)

    diffs: list[int] = []
    peaks: list[int] = []
    for _ in range(steps):
        rss_before = proc.memory_info().rss
        model(sample)
        rss_after = proc.memory_info().rss
        diffs.append(max(0, rss_after - rss_before))
        peaks.append(max(0, rss_after - rss0))

    return MemoryMetrics(
        avg_mib=_bytes_to_mib(float(np.mean(diffs))),
        peak_mib=_bytes_to_mib(float(np.max(peaks))),
        reserved_mib=float("nan"),
        device="cpu",
    )


#| export
def compute_memory(
    model: torch.nn.Module,                  # model to benchmark
    sample: torch.Tensor,                    # input tensor (with batch dimension)
    *,
    device: str | torch.device = "cpu",      # device to run on
    warmup: int = 10,                        # warmup iterations
    steps: int = 100,                        # measurement iterations
) -> MemoryMetrics:
    """Measure memory usage on specified device."""
    device_str = str(device)
    if device_str == "cuda" or (device_str == "auto" and torch.cuda.is_available()):
        if not torch.cuda.is_available():
            warnings.warn("CUDA requested but not available – falling back to CPU")
            return _cpu_metrics(model, sample, warmup=warmup, steps=steps)
        return _gpu_metrics(model, sample, warmup=warmup, steps=steps)
    return _cpu_metrics(model, sample, warmup=warmup, steps=steps)


#| export
def compute_memory_multi(
    model: torch.nn.Module,                                # model to benchmark
    sample: torch.Tensor,                                  # input tensor (with batch dimension)
    *,
    devices: Sequence[str | torch.device] | None = None,   # devices to benchmark (default: cpu + cuda)
    warmup: int = 10,                                      # warmup iterations
    steps: int = 100,                                      # measurement iterations
) -> dict[str, MemoryMetrics]:
    """Measure memory on multiple devices."""
    return _run_on_devices(
        compute_memory, model, sample, devices,
        nan_factory=_nan_memory_metrics,
        metric_name="Memory",
        warmup=warmup,
        steps=steps,
    )

In [ ]:
show_doc(MemoryMetrics)

In [ ]:
show_doc(compute_memory)

In [ ]:
show_doc(compute_memory_multi)

In [ ]:
#| hide
from fastcore.test import *

import torch, torch.nn as nn
_m = nn.Linear(10, 5)
_x = torch.randn(1, 10)
_mem = compute_memory(_m, _x, warmup=2, steps=5)
assert isinstance(_mem, MemoryMetrics)
assert _mem.avg_mib >= 0

---

## See Also

- [Speed](speed.html) — Latency measurement
- [Benchmark](../analysis/benchmark.html) — Unified API